In [2]:
import os
import argparse
import numpy as np
import pandas as pd
import cv2
import joblib
import warnings

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ==============================================================================
# ENGINE SELECTION (Carregado do motor principal)
# ==============================================================================
try:
    import cupy as cp
    from cupyx.scipy import ndimage as cp_ndimage
    CUPY_AVAILABLE = True
except ImportError:
    CUPY_AVAILABLE = False

# Importaremos a classe ARQUE_Math base do repositório (Assumindo que o arquivo 
# principal foi salvo com o nome que sugerimos no benchmark, ou copiamos as classes aqui)
# Para evitar dependências externas no App, estou instanciando os Numba cores aqui
from numba import jit, prange

@jit(nopython=True, parallel=True, fastmath=True)
def numba_batch_atr_grid(ch, cv, alphas, betas):
    n_configs = len(alphas); rows, cols = ch.shape; size = ch.size
    lh = np.std(ch); lv = np.std(cv)
    if lh < 1e-9: lh = 1e-9
    if lv < 1e-9: lv = 1e-9
    scores = np.zeros(n_configs, dtype=np.float64)
    for k in prange(n_configs):
        a = alphas[k]; b = betas[k]
        beta_lv, alpha_lh = b * lv, a * lh
        beta_lh, alpha_lv = b * lh, a * lv
        count = 0
        for r in range(rows):
            for c in range(cols):
                val_ch = ch[r, c]; val_cv = cv[r, c]
                if (val_cv >= beta_lv) and (val_ch < alpha_lh): count += 1
                if (val_ch >= beta_lh) and (val_cv < alpha_lv): count += 1
        scores[k] = (count / 2.0) / size
    return scores

@jit(nopython=True, fastmath=True)
def numba_stats_moments(arr_flat):
    n = len(arr_flat)
    if n == 0: return 0.0, 0.0, 0.0, 0.0
    mean_val = 0.0
    for i in range(n): mean_val += arr_flat[i]
    mean_val /= n
    m2, m3, m4 = 0.0, 0.0, 0.0
    for i in range(n):
        val = arr_flat[i] - mean_val
        val2 = val * val
        m2 += val2; m3 += val2 * val; m4 += val2 * val2
    var = m2 / n; std_val = np.sqrt(var)
    if m2 == 0: return -3.0, 0.0, std_val, mean_val
    else: return (m4 / n) / (var**2) - 3.0, (m3 / n) / (std_val**3), std_val, mean_val

@jit(nopython=True, fastmath=True)
def numba_pairwise_stats(mat):
    rows, cols = mat.shape
    if cols < 2: return 0.0, 0.0
    n_pairs = rows * (cols - 1); sum_prod = 0.0
    for r in range(rows):
        for c in range(cols - 1): sum_prod += mat[r, c] * mat[r, c+1]
    mean_prod = sum_prod / n_pairs; sum_sq_diff = 0.0
    for r in range(rows):
        for c in range(cols - 1):
            diff = (mat[r, c] * mat[r, c+1]) - mean_prod
            sum_sq_diff += diff * diff
    return mean_prod, np.sqrt(sum_sq_diff / n_pairs)

@jit(nopython=True, parallel=True, fastmath=True)
def numba_single_atr(ch, cv, alpha, beta):
    lh = np.std(ch); lv = np.std(cv)
    if lh < 1e-9: lh = 1e-9
    if lv < 1e-9: lv = 1e-9
    rows, cols = ch.shape
    beta_lv, alpha_lh = beta * lv, alpha * lh
    beta_lh, alpha_lv = beta * lh, alpha * lv
    cnt = 0
    for i in prange(rows):
        for j in range(cols):
            vch = ch[i, j]; vcv = cv[i, j]
            if (vcv >= beta_lv) and (vch < alpha_lh): cnt += 1
            if (vch >= beta_lh) and (vcv < alpha_lv): cnt += 1
    return cnt / 2.0 / ch.size

from scipy import ndimage
class ARQUE_Math_Numba_V2:
    def calculate_log1p_abs_curvatures(self, img, Dir='hv'):
        if Dir =='hv': kh=np.array([[1,-2,1]]); kv=np.array([[1],[-2],[1]])
        else: kh=np.array([[1,0,0],[0,-2,0],[0,0,1]]); kv=np.array([[0,0,1],[0,-2,0],[1,0,0]])
        img = img.astype(np.float32)
        ch = np.log1p(np.abs(ndimage.convolve(img, kh, mode='reflect')))
        cv = np.log1p(np.abs(ndimage.convolve(img, kv, mode='reflect')))
        return ch, cv
    def get_atr_single(self, ch, cv, alpha, beta): return numba_single_atr(ch, cv, float(alpha), float(beta))
    def get_nss_features(self, ch, cv):
        f = []
        for m in [ch, cv]:
            k, s, std, mean = numba_stats_moments(m.flatten())
            f.extend([k, s, std, mean])
        for m in [ch, cv]:
            mean_p, std_p = numba_pairwise_stats(m)
            f.extend([mean_p, std_p])
        return f

class ARQUE_Math_CuPy:
    def calculate_log1p_abs_curvatures(self, img, Dir='hv'):
        img_gpu = cp.asarray(img, dtype=cp.float32)
        if Dir == 'hv':
            kh, kv = cp.array([[1, -2, 1]], dtype=cp.float32), cp.array([[1], [-2], [1]], dtype=cp.float32)
        else:
            kh = cp.array([[1, 0, 0], [0, -2, 0], [0, 0, 1]], dtype=cp.float32)
            kv = cp.array([[0, 0, 1], [0, -2, 0], [1, 0, 0]], dtype=cp.float32)
        ch = cp.log1p(cp.abs(cp_ndimage.convolve(img_gpu, kh, mode='reflect')))
        cv = cp.log1p(cp.abs(cp_ndimage.convolve(img_gpu, kv, mode='reflect')))
        return ch, cv 

    def get_atr_single(self, ch, cv, alpha, beta):
        size = ch.size
        lh, lv = cp.maximum(cp.std(ch), 1e-9), cp.maximum(cp.std(cv), 1e-9)
        beta_lv, alpha_lh = beta * lv, alpha * lh
        beta_lh, alpha_lv = beta * lh, alpha * lv
        c1 = (cv >= beta_lv) & (ch < alpha_lh)
        c2 = (ch >= beta_lh) & (cv < alpha_lv)
        cnt = cp.sum(c1) + cp.sum(c2)
        return float((cnt / (2.0 * size)).get())

    def get_nss_features(self, ch, cv):
        f = []
        for m in [ch, cv]:
            mean_val, std_val = cp.mean(m), cp.std(m)
            if std_val == 0: skew_val, kurt_val = 0.0, -3.0
            else:
                m_centered = m - mean_val
                var = std_val ** 2
                skew_val = cp.mean(m_centered ** 3) / (std_val ** 3)
                kurt_val = cp.mean(m_centered ** 4) / (var ** 2) - 3.0
            f.extend([float(kurt_val.get()), float(skew_val.get()), float(std_val.get()), float(mean_val.get())])
        for m in [ch, cv]:
            if m.shape[1] < 2: mean_p, std_p = 0.0, 0.0
            else:
                prod = m[:, :-1] * m[:, 1:]
                mean_p, std_p = cp.mean(prod), cp.std(prod)
            f.extend([float(mean_p.get()), float(std_p.get())])
        return f

# ==============================================================================
# PIPELINE DE INFERÊNCIA DO APP
# ==============================================================================
def process_image(img_path, math_core, models_dict):
    """
    Processa uma imagem e retorna os scores do Ensemble AQI e o perfil de ruído.
    """
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None

    # 1. Extração de Features Geométricas (Uma única vez para todos os modelos)
    ch, cv = math_core.calculate_log1p_abs_curvatures(img)
    nss = math_core.get_nss_features(ch, cv)
    
    results = {}
    
    # 2. Avaliação em cada Modelo (Escalas normalizadas [0-100] pelo export_models.py)
    for model_name, payload in models_dict.items():
        params = payload['optimized_params']
        clf = payload['clf']
        svrs = payload['svrs']
        le = payload['label_encoder']
        
        # Reconstrói a grade de ATR específica daquele modelo
        sorted_keys = sorted(params.keys())
        atrs_all = []
        for k in sorted_keys:
            pk = params[k]
            atrs_all.append(math_core.get_atr_single(ch, cv, pk['alpha'], pk['beta']))
            
        # Classificador (Diagnóstico de Ruído)
        feat_vector = np.array([atrs_all + nss])
        probs = clf.predict_proba(feat_vector)[0]
        
        # Captura o Top 1 Diagnóstico para o usuário
        max_idx = np.argmax(probs)
        top_distortion = le.inverse_transform([max_idx])[0]
        confidence = probs[max_idx]
        
        # Fusão dos SVRs Especialistas
        val = 0.0
        w_sum = 0.0
        for idx, prob in enumerate(probs):
            if prob < 0.01: continue
            c_name = le.inverse_transform([idx])[0]
            if c_name in svrs:
                pk = params[c_name]
                atr_spec = math_core.get_atr_single(ch, cv, pk['alpha'], pk['beta'])
                feat_svr = np.array([[atr_spec] + nss])
                
                # A predição JÁ VEM na escala AQI [0-100] porque treinamos assim
                val += prob * svrs[c_name].predict(feat_svr)[0]
                w_sum += prob
                
        final_aqi = val / w_sum if w_sum > 0 else 50.0
        
        results[model_name] = {
            'aqi_score': max(0.0, min(100.0, final_aqi)), # Grampeia entre 0 e 100
            'distortion': top_distortion,
            'confidence': confidence * 100.0
        }
        
    return results

if __name__ == "__main__":

    parser = argparse.ArgumentParser(description="ARQUE Image Quality Assessment (AQI Scale)")
    
    # Descobre dinamicamente o caminho correto (Plano A: GitHub/Relativo | Plano B: Seu PC local)
    if os.path.exists('./test_images'):
        default_path = './test_images'
    elif os.path.exists('app/test_images'):
        default_path = 'app/test_images'
    else:
        default_path = r"C:\Users\User\NOTEBOOKS-DELL\IMAGE_QUALITY\app\test_images"

    parser.add_argument('-i', '--input', required=False, 
                        default=default_path, 
                        help='Caminho da pasta contendo as imagens')
    parser.add_argument('-o', '--output', default='relatorio_qualidade.csv', help='Nome do CSV de saída')
    
    parser.add_argument('--gpu', action='store_true', default=False, help='Ativa aceleração via GPU (CuPy)')
    
    args, unknown = parser.parse_known_args()
        
    # 1. Configura a Engine (CPU/GPU)
    if args.gpu and CUPY_AVAILABLE:
        print(">>> Iniciando motor ARQUE: [GPU - CuPy] <<<")
        math_core = ARQUE_Math_CuPy()
    else:
        print(">>> Iniciando motor ARQUE: [CPU - Numba] <<<")
        math_core = ARQUE_Math_Numba_V2()

    # (O resto do código segue exatamente igual para carregar os modelos e fazer a varredura...)

    # 2. Carrega os modelos pre-treinados (.joblib)
    models_dir = r"C:\Users\User\NOTEBOOKS-DELL\IMAGE_QUALITY\app\models"
    model_files = [f for f in os.listdir(models_dir) if f.endswith('.joblib')]
    
    if not model_files:
        print(f"[Erro Crítico] Nenhum arquivo .joblib encontrado em {models_dir}")
        print("Por favor, rode o script 'export_models.py' no repositório de benchmark primeiro.")
        sys.exit(1)

    print(f"\n[Info] Carregando {len(model_files)} modelos ARQUE especialistas...")
    loaded_models = {}
    for mf in model_files:
        path = os.path.join(models_dir, mf)
        payload = joblib.load(path)
        name = payload['dataset_origin']
        loaded_models[name] = payload
        print(f"       -> Modelo [{name}] carregado com sucesso.")

    # 3. Varredura do Diretório
    valid_exts = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')
    img_files = [f for f in os.listdir(args.input) if f.lower().endswith(valid_exts)]
    
    if not img_files:
        print(f"\n[Erro] Nenhuma imagem encontrada em: {args.input}")
        sys.exit(1)

    print(f"\n[Processamento] Analisando {len(img_files)} imagens...")
    
    report_data = []
    
    for i, img_name in enumerate(img_files):
        img_path = os.path.join(args.input, img_name)
        res = process_image(img_path, math_core, loaded_models)
        
        if res is None:
            print(f"       [{i+1}/{len(img_files)}] Erro ao ler: {img_name} (Pulando)")
            continue
            
        # Consolida os dados para o CSV
        row = {'Image': img_name}
        scores = [] # Nova lista para guardar as notas para o consenso
        
        for m_name, metrics in res.items():
            row[f'AQI_{m_name}'] = metrics['aqi_score']
            row[f'Noise_{m_name}'] = metrics['distortion']
            row[f'Conf_{m_name}(%)'] = metrics['confidence']
            scores.append(metrics['aqi_score'])
            
        # =======================================================
        # LÓGICA "2 DE 3" (CONSENSO DE VIZINHANÇA)
        # =======================================================
        scores.sort() # Ordena do menor para o maior
        
        # Se temos exatamente 3 modelos, achamos os 2 que mais concordam
        if len(scores) == 3:
            dist_1_2 = abs(scores[0] - scores[1])
            dist_2_3 = abs(scores[1] - scores[2])
            
            if dist_1_2 < dist_2_3:
                # Os dois menores concordam mais. O maior é um outlier e será ignorado.
                aqi_final = (scores[0] + scores[1]) / 2.0
            else:
                # Os dois maiores concordam mais. O menor é um outlier e será ignorado.
                aqi_final = (scores[1] + scores[2]) / 2.0
        else:
            # Fallback caso rode com uma quantidade diferente de modelos
            aqi_final = np.median(scores) 
            
        row['AQI_FINAL_CONSENSO'] = aqi_final
        report_data.append(row)
        
        print(f"       [{i+1}/{len(img_files)}] {img_name:20s} | Final AQI: {row['AQI_FINAL_CONSENSO']:.2f}/100")

    # 4. Salvar CSV
    df = pd.DataFrame(report_data)
    
    # Reorganiza colunas para deixar o AQI Final logo no começo
    cols = ['Image', 'AQI_FINAL_CONSENSO'] + [c for c in df.columns if c not in ['Image', 'AQI_FINAL_CONSENSO']]
    df = df[cols]
    
    # Arredonda valores decimais
    df = df.round(2)
    
    df.to_csv(args.output, index=False)
    print(f"\n[Concluído] Relatório exportado com sucesso para: {os.path.abspath(args.output)}")
    print("O AQI (ARQUE Quality Index) varia de 0 (Péssimo) a 100 (Perfeito).")
   

>>> Iniciando motor ARQUE: [CPU - Numba] <<<

[Info] Carregando 3 modelos ARQUE especialistas...
       -> Modelo [CSIQ] carregado com sucesso.
       -> Modelo [LIVE] carregado com sucesso.
       -> Modelo [TID2013] carregado com sucesso.

[Processamento] Analisando 120 imagens...
       [1/120] sample_01_CSIQ_1600.blur.2.png | Final AQI: 70.70/100
       [2/120] sample_01_LIVE_img54.bmp | Final AQI: 33.09/100
       [3/120] sample_02_CSIQ_monument.AWGN.3.png | Final AQI: 61.00/100
       [4/120] sample_02_CSIQ_woman.blur.3.png | Final AQI: 38.96/100
       [5/120] sample_03_TID_i21_23_2.bmp | Final AQI: 90.37/100
       [6/120] sample_03_TID_i23_16_5.bmp | Final AQI: 98.14/100
       [7/120] sample_04_LIVE_img119.bmp | Final AQI: 94.31/100
       [8/120] sample_04_TID_i07_13_2.bmp | Final AQI: 91.17/100
       [9/120] sample_05_TID_i12_17_2.bmp | Final AQI: 91.66/100
       [10/120] sample_05_TID_i22_11_3.bmp | Final AQI: 59.03/100
       [11/120] sample_06_TID_i14_24_4.bmp | Final 